# Классификация изображений Conv2D — версия 2.0

    **Цель:** Обучить классификатор и правильно подготовить изображения.

    **Алгоритм:** Свёрточная сеть классифицирует всё изображение; координаты объектов она не предсказывает. Историческое имя файла сохранено.

    Учебная модернизация 2026-10-01. Код адаптирован с Python 3.7 на Python 3.12. Исходные алгоритмы сохранены; историческая
    версия доступна в Git на коммите `a842320`. Исправления учебных ошибок описаны
    в README. Это учебное пособие, не промышленная система и не benchmark.

    По умолчанию: CPU, короткий запуск, синтетические демонстрационные данные,
    без загрузок из сети. Реальные данные включаются отдельно: `MLLAB_DATA_SOURCE`.
    Синтетические результаты нельзя выдавать за качество на реальных данных.
    После изменения настроек перезапустите ядро и выполните ячейки сверху вниз.
    Окружение и структура данных описаны в `docs/ENVIRONMENTS.md` и `docs/DATA.md`.


In [ ]:
from pathlib import Path
import os, sys
start = Path(os.environ.get('MLLAB_ROOT', str(Path.cwd()))).resolve()
root = next((p for p in (start, *start.parents) if (p / 'ml_lab' / 'runtime.py').is_file()), None)
if root is None:
    raise FileNotFoundError('Откройте папку репозитория или задайте MLLAB_ROOT.')
sys.path.insert(0, str(root))
from ml_lab.runtime import settings
cfg = settings('object-detection-conv2d')


## Данные
Размер массива — (N, height, width, channels), PIL.resize принимает (width, height).


In [ ]:
from ml_lab.runtime import tensorflow_setup, demo_images, resize_gray
tf = tensorflow_setup(cfg)
from tensorflow.keras import layers, Sequential
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import matplotlib.pyplot as plt
height, width = 32, 64
if cfg.source == 'demo':
    images, labels = demo_images(160, height, width, seed=cfg.seed)
    names = ['Демо-класс 0', 'Демо-класс 1']
elif cfg.source == 'local':
    folders = sorted(p for p in (cfg.data / 'images').iterdir() if p.is_dir())
    names = [p.name for p in folders]
    pairs = [(p, i) for i, folder in enumerate(folders) for p in sorted(folder.iterdir()) if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}]
    if len(names) < 2 or not pairs:
        raise ValueError('Нужны минимум две папки классов в data/images.')
    images = np.stack([resize_gray(p, height, width) for p, _ in pairs])
    labels = np.array([label for _, label in pairs])
else:
    raise ValueError('Для классификатора используйте demo или local')
assert images.shape[1:] == (height, width, 1)
x_train, x_rest, y_train, y_rest = train_test_split(images, labels, test_size=.4, stratify=labels, random_state=cfg.seed)
x_val, x_test, y_val, y_test = train_test_split(x_rest, y_rest, test_size=.5, stratify=y_rest, random_state=cfg.seed)
print('train/val/test:', x_train.shape, x_val.shape, x_test.shape)


## Исходная архитектура CNN
Число выходов берётся из классов; модель создаётся заново для независимого опыта.


In [ ]:
def build_model():
    return Sequential([
        layers.Input(shape=(height, width, 1)),
        layers.Conv2D(16, 3, padding='same', activation='relu'), layers.BatchNormalization(), layers.SpatialDropout2D(.3),
        layers.Conv2D(32, 3, padding='same', activation='relu'), layers.BatchNormalization(), layers.MaxPooling2D(2),
        layers.Conv2D(64, 3, padding='same', activation='relu'), layers.BatchNormalization(), layers.SpatialDropout2D(.5),
        layers.Flatten(), layers.Dense(128, activation='relu'), layers.BatchNormalization(),
        layers.Dense(128, activation='relu'), layers.BatchNormalization(), layers.Dense(64, activation='relu'),
        layers.Dropout(.5), layers.Dense(len(names), activation='softmax')])
model = build_model()
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=.0001), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history = model.fit(x_train, y_train, epochs=1 if cfg.quick else 50, batch_size=16 if cfg.quick else 128, validation_data=(x_val, y_val), verbose=0)
plt.plot(history.history['loss'], label='train'); plt.plot(history.history['val_loss'], label='validation'); plt.legend(); plt.show()


## Независимая оценка
Никакого требования «90%» без реального теста и протокола.


In [ ]:
predictions = model.predict(x_test, verbose=0).argmax(axis=1)
print(classification_report(y_test, predictions, labels=np.arange(len(names)), target_names=names, zero_division=0))
print('Confusion matrix:', confusion_matrix(y_test, predictions, labels=np.arange(len(names))))
model.save(cfg.output / 'classifier.keras')


## Результат и самостоятельная работа

    Выполненные ячейки показывают формы данных, обучение и результат алгоритма.
    Успешный короткий запуск проверяет работоспособность, а не достижение заданной точности.

    1. Объясните отличие классификации от детекции.
2. Проверьте порядок width/height.
3. Сравните batch_size и learning_rate на одной фиксированной разбивке.
4. Почему test нельзя использовать для выбора модели?

    Перед выводом о качестве используйте отдельные train/validation/test,
    документируйте источник данных и сравнивайте эксперименты при одинаковых настройках.
